In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import seaborn as sns
import os
import itertools

from matplotlib.collections import PatchCollection
from matplotlib.backends.backend_pdf import PdfPages
from pathlib import Path
from datetime import datetime
from sklearn.preprocessing import MinMaxScaler
from minisom import MiniSom

SESSION_TIMESTAMP = datetime.now().strftime("%Y-%m-%d_%H-%M-%S")

In [2]:
pd.set_option('display.max_columns', None)
sns.set_theme(style="whitegrid")

In [3]:
# ------------------------- SOM Hyperparameter Konfiguration -------------------------
# Hier können die Parameter für das SOM-Training angepasst werden.

# ------------------------- Dimensionen der Karte (Grid) -------------------------
SOM_X = 3
SOM_Y = 3

# ------------------------- Training Parameter -------------------------
SOM_ITERATIONS = 500       # Anzahl der Trainings-Iterationen
SOM_SIGMA = 0.5             # Start-Radius der Nachbarschaftsfunktion
SOM_LEARNING_RATE = 0.5     # Lernrate
SOM_RANDOM_SEED = 42        # Reproduzierbarkeit

# ------------------------- Topologie & Funktionen -------------------------
SOM_TOPOLOGY = 'hexagonal'             # 'rectangular' oder 'hexagonal'
SOM_NEIGHBORHOOD = 'gaussian'          # 'gaussian', 'mexican_hat', 'bubble', 'triangle'
SOM_ACTIVATION = 'euclidean'           # 'euclidean', 'cosine', 'manhattan', 'chebyshev'

print("SOM Konfiguration geladen:")
print(f"Größe: {SOM_X}x{SOM_Y}")
print(f"Iterationen: {SOM_ITERATIONS}")
print(f"Sigma: {SOM_SIGMA}, LR: {SOM_LEARNING_RATE}")

SOM Konfiguration geladen:
Größe: 3x3
Iterationen: 500
Sigma: 0.5, LR: 0.5


In [4]:
# ------------------------- Konfiguration -------------------------
# MANUAL für manuelle Feature-Auswahl
# LOOP für automatische Feature-Kombination

EXECUTION_MODE = os.environ.get('SOM_MODE', 'MANUAL')


# ------------------------- Attribut-Auswahl (verpflichtend) -------------------------
# Die Hauptionen sollten immer dabei sein (FIXED_BASE_FEATURES)

FIXED_BASE_FEATURES = [
    "Na_in_mmol/L",
    "Mg_in_mmol/L",
    "Ca_in_mmol/L",
    "Cl_in_mmol/L",
    "SO4_in_mmol/L",
    "HCO3_in_mmol/L",
    "total_dissolved_solids_in_mmol/L",
]


# ------------------------- Attribut-Auswahl (manuell) ------------------------- 
# Alle hier einkommentierten Features werden ZUSÄTZLICH zur Basis genutzt (OPTIONAL_FEATURES_POOL).

OPTIONAL_FEATURES_POOL = [
    # --------- Physikalische Parameter -----------
    "temperature_in_c",
    "pH",
    "electrical_conductivity_25c_in_uS/cm",
    "redox_potential_in_mV",

    # --------- Weitere Ionen / Spurenelemente ---------
    "K_in_mmol/L",
    "NO3_in_mmol/L",
    "Li_in_mmol/L",
    "Fe_in_mmol/L",
    "Mn_in_mmol/L",
    "HS_in_mmol/L",
    "O2_in_mmol/L",
    "Sr_in_umol/L",
    "NH4_in_umol/L",
    "F_in_umol/L",
    "H2SiO3_in_umol/L",
    
    # --------- Metadaten (nur numerisch nutzen) ---------
    "depth_bgl_in_m",
    # "stratigraphic_period",
    # "rock_type"
]


print(f"Modus: {EXECUTION_MODE}")
print(f"Fixed Base ({len(FIXED_BASE_FEATURES)}): {FIXED_BASE_FEATURES}")
print(f"Optional Pool ({len(OPTIONAL_FEATURES_POOL)}): {OPTIONAL_FEATURES_POOL}")

Modus: LOOP
Fixed Base (7): ['Na_in_mmol/L', 'Mg_in_mmol/L', 'Ca_in_mmol/L', 'Cl_in_mmol/L', 'SO4_in_mmol/L', 'HCO3_in_mmol/L', 'total_dissolved_solids_in_mmol/L']
Optional Pool (16): ['temperature_in_c', 'pH', 'electrical_conductivity_25c_in_uS/cm', 'redox_potential_in_mV', 'K_in_mmol/L', 'NO3_in_mmol/L', 'Li_in_mmol/L', 'Fe_in_mmol/L', 'Mn_in_mmol/L', 'HS_in_mmol/L', 'O2_in_mmol/L', 'Sr_in_umol/L', 'NH4_in_umol/L', 'F_in_umol/L', 'H2SiO3_in_umol/L', 'depth_bgl_in_m']


In [5]:
# ------------------------- Basisverzeichnis (aktuelles Notebook-Verzeichnis) -------------------------
base_dir = Path.cwd()


# ------------------------- Suche nach dem neuesten Preprocessing-Ordner -------------------------
preprocessing_root = base_dir.parent.parent / "3.1_Preprocessing" / "Preprocessing"
timestamp_folders = [f for f in preprocessing_root.iterdir() if f.is_dir()]
if not timestamp_folders:
    raise FileNotFoundError(f"Keine Preprocessing-Ordner in {preprocessing_root} gefunden.")

latest_folder = max(timestamp_folders, key=lambda f: f.stat().st_mtime)
print(f"Verwendeter Preprocessing-Ordner: {latest_folder.name}")


# ------------------------- Zeitstempel finden (neuesten Ordner) -------------------------
try:
    prep_ts = datetime.strptime(latest_folder.name, "%Y-%m-%d_%H-%M-%S")
except ValueError:
    print("Warnung: Preprocessing-Ordnername entspricht nicht dem Schema. Nutze Dateisystem-Zeit.")
    prep_ts = datetime.fromtimestamp(latest_folder.stat().st_mtime)


# ------------------------- Lade Preprocessed Data -------------------------
input_path_prep = latest_folder / "Preprocessed_SOM_Ready.csv"
df_full = pd.read_csv(input_path_prep, low_memory=False)

print(f"Daten erfolgreich aus 3.1_Preprocessing geladen! Shape: {df_full.shape}")

Verwendeter Preprocessing-Ordner: 2026-01-17_15-04-34
Daten erfolgreich aus 3.1_Preprocessing geladen! Shape: (94264, 51)


<div class="alert alert-info">
    <b>Temperature Analysis & Hexagonal SOM</b><br><br>
    <b>Datenquelle:</b><br>
    - Preprocessed Data: Ionen (Log + Gaussian Scaling) + Temperatur (Cleaned)<br>
    <br>
    <b>Filter:</b><br>
    - <b>Nur Temperaturen > 10 °C</b> werden berücksichtigt.
    <br>
    <b>Ziel:</b><br>
    - Clustering mittels SOM (Hexagonal).
    - Analyse der Temperaturverteilung innerhalb der Cluster.
    - Export als PDF Report.
</div>

In [6]:
# ------------------------- Features-Normierung aus 3.1 -------------------------

def get_training_features(user_selection, df_columns):
    training_features = []
    mapping_report = []
    
    for user_col in user_selection:
        found = False
        # ------------------------- Suche nach transformierter Version (_gauss) -------------------------
        candidates = [c for c in df_columns if c.startswith(user_col) and "_gauss" in c]
        if candidates:
            # ------------------------- Priorisiere _log_gauss -------------------------
            best_match = next((c for c in candidates if "_log_gauss" in c), candidates[0])
            training_features.append(best_match)
            mapping_report.append(f"{user_col} -> {best_match}")
            found = True
        else:
            # ------------------------- Exakter Match -------------------------
            if user_col in df_columns:
                training_features.append(user_col)
                mapping_report.append(f"{user_col} -> {user_col} (Raw)")
                found = True
        
        if not found:
             # ------------------------- Fallback -------------------------
             fuzzy = [c for c in df_columns if c.startswith(user_col) and "_gauss" in c]
             if fuzzy:
                 best_match = fuzzy[0]
                 training_features.append(best_match)
                 mapping_report.append(f"{user_col} -> {best_match} (Fuzzy)")
                 found = True
                 
        if not found:
             print(f"Feature '{user_col}' nicht gefunden! Wird ignoriert.")
             
    return training_features, mapping_report

In [7]:
# ------------------------- Plotting -------------------------
def plot_hex_map_to_axis(ax, data_matrix, title, cmap='viridis', label_suffix='', annotate=False):
    sy, sx = data_matrix.shape
    ax.set_aspect('equal')
    patches = []
    colors = []
    for y in range(sy):
        for x in range(sx):
            val = data_matrix[y, x]
            if pd.isna(val): continue
            offset = 0.5 if y % 2 != 0 else 0.0
            center_x = x + offset
            center_y = y * (np.sqrt(3) / 2)
            radius = 1 / np.sqrt(3)
            hex_poly = mpatches.RegularPolygon((center_x, center_y), numVertices=6, radius=radius, orientation=np.radians(30), edgecolor='k', linewidth=0.5)
            patches.append(hex_poly)
            colors.append(val)
            if annotate:
                ax.text(center_x, center_y, f"{val:.1f}", ha='center', va='center', fontsize=7, fontweight='bold')
    if not patches: return
    collection = PatchCollection(patches, cmap=cmap, alpha=0.9)
    collection.set_array(np.array(colors))
    ax.add_collection(collection)
    ax.set_xlim(-0.5, sx + 0.5)
    ax.set_ylim(-0.5, sy * (np.sqrt(3)/2) + 0.5)
    ax.axis('off')
    ax.set_title(title, fontsize=12)
    return collection

In [8]:
def create_cluster_page(pdf_obj, df_cluster, cluster_id, train_cols, som_x, som_y, df_run):
    
    # ------------------------- Seitengröße und Layout ----------------------------

    # Rows:
    # 0. Header -------------------------------------------------- 12%
    # 1. Physikalische Parameter --------------------------------- 13%
    # 2. Gesteinsschnitt ----------------------------------------- 20%
    # 3. SPACER (gegen Überlappung) ------------------------------ 5%
    # 4. Chemische zusammensetzung (Originalwerte) --------------- 45%
    # 5. Stratigraphie (Text) ------------------------------------ 5%
    
    fig = plt.figure(figsize=(11.7, 16.5)) # A4 Format
    
    # ------------------------- Gleiche Ratios -------------------------
    gs = fig.add_gridspec(6, 1, height_ratios=[0.12, 0.15, 0.20, 0.05, 0.45, 0.03], hspace=0.6)
    
    
    # ------------------------- Header Area  -------------------------
    gs_header = gs[0].subgridspec(2, 2, width_ratios=[0.55, 0.45], height_ratios=[0.4, 0.6])
    
    # ------------------------- Titel (oben links) -------------------------
    ax_title = fig.add_subplot(gs_header[0, 0])
    ax_title.axis('off')
    ax_title.text(0.0, 0.3, f"Cluster: {cluster_id} (N={len(df_cluster)})", 
                  ha='left', va='center', fontsize=20, fontweight='bold')
                  

    #---------------------------- MiniMap (Oben rechts) -------------------------
    ax_map = fig.add_subplot(gs_header[1, 1])
    ax_map.set_aspect('equal')
    ax_map.axis('off')
    ax_map.set_title(f"Position (SOM {som_x}x{som_y})", fontsize=10)
    
    # ------------------------- MiniMap Logik -------------------------
    cy_idx = cluster_id[0] - 1
    cx_idx = cluster_id[1] - 1
    
    for y in range(som_y):
        for x in range(som_x):
            is_active = (x == cx_idx and y == cy_idx)
            
            offset = 0.5 if y % 2 != 0 else 0.0
            center_x = x + offset
            center_y = y * (np.sqrt(3) / 2)
            radius = 1 / np.sqrt(3)
            
            col = 'red' if is_active else '#EEEEEE'
            alpha = 1.0 if is_active else 0.3
            
            hex_poly = mpatches.RegularPolygon(
                (center_x, center_y),
                numVertices=6,
                radius=radius,
                orientation=np.radians(30), 
                facecolor=col,
                edgecolor='k',
                linewidth=0.5,
                alpha=alpha)
            ax_map.add_patch(hex_poly)
            

    # ------------------------- Fit -------------------------
    ax_map.set_xlim(-0.5, som_x + 0.5)
    ax_map.set_ylim(-0.5, som_y * (np.sqrt(3)/2) + 0.5)
    ax_map.invert_yaxis()
    
    # ------------------------- 6 Haxgoene der Hauptionen darstellen -------------------------
    main_ions = ['Na_in_mmol/L', 'Mg_in_mmol/L', 'Ca_in_mmol/L', 'Cl_in_mmol/L', 'SO4_in_mmol/L', 'HCO3_in_mmol/L']
    labels_ions = ['Na', 'Mg', 'Ca', 'Cl', 'SO4', 'HCO3']
    
    ax_ions = fig.add_subplot(gs_header[1, 0])
    ax_ions.axis('off')


    for i, ion_col_raw in enumerate(main_ions):
        
        # ------------------------- Farbsynchronisierung Hexagone und Chemischer Schnitt -------------------------
        target_col = ion_col_raw 
        for tc in train_cols:
            if tc.startswith(ion_col_raw):
                target_col = tc
                break
        
        # ------------------------ Normalisierung auf Graphschnitt -------------------------
        if target_col in df_run.columns:
            agg = df_run.groupby(['som_x', 'som_y'])[target_col].mean()
            vmin = agg.min()
            vmax = agg.max()
            
            # ------------------------- Schnitt für das aktuelle Cluster nehmen -------------------------
            val_mean = df_cluster[target_col].mean()
        else:
            vmin, vmax = 0, 1
            val_mean = 0
            
        norm = plt.Normalize(vmin, vmax)
        cmap = plt.cm.viridis
        col = cmap(norm(val_mean))
        
        # ------------------------- Textfarbe an Hand von Hintergrund bestimmen (schwarz / weiß) -------------------------
        normalized_val = norm(val_mean)
        text_col = 'white' if normalized_val < 0.6 else 'black'
        
        spacing = 2.8
        center_x = i * spacing
        center_y = 0
        radius = 1.3
        
        hex_poly = mpatches.RegularPolygon((center_x, center_y), numVertices=6, radius=radius,
                                           orientation=np.radians(30), 
                                           facecolor=col, edgecolor='k', linewidth=1)
        ax_ions.add_patch(hex_poly)
        
        ax_ions.text(center_x, center_y, labels_ions[i], ha='center', va='center', fontweight='bold', fontsize=11, color=text_col)
        


    # ------------------------- Anordnung -------------------------
    ax_ions.set_xlim(-1.5, 15.5)
    ax_ions.set_ylim(-1.5, 1.5)
    ax_ions.set_aspect('equal')


    if len(df_cluster) < 2:
         plt.close(fig)
         return

    # ------------------------- Physikalische Parameter (Reihe 1) -------------------------
    gs_phys = gs[1].subgridspec(1, 3, wspace=0.3)
    
    # ------------------------- 1. Temperatur
    ax_temp = fig.add_subplot(gs_phys[0])
    if 'temperature_in_c' in df_cluster.columns and df_cluster['temperature_in_c'].notna().sum() > 1:
        sns.histplot(data=df_cluster, x='temperature_in_c', kde=True, ax=ax_temp, color='orange')
        ax_temp.set_title(f"Temp [°C] (Durchschnitt: {df_cluster['temperature_in_c'].mean():.1f})", fontsize=9)
        ax_temp.set_xlabel("")
        
    # ------------------------- 2. pH
    ax_ph = fig.add_subplot(gs_phys[1])
    if 'pH' in df_cluster.columns and df_cluster['pH'].notna().sum() > 1:
        sns.histplot(data=df_cluster, x='pH', kde=True, ax=ax_ph, color='purple')
        ax_ph.set_title(f"pH [-] (Durchschnitt: {df_cluster['pH'].mean():.1f})", fontsize=9)
        ax_ph.set_xlabel("")
        
    # -------------------------3. Tiefe
    ax_depth = fig.add_subplot(gs_phys[2])
    if 'depth_bgl_in_m' in df_cluster.columns and df_cluster['depth_bgl_in_m'].notna().sum() > 1:
        sns.histplot(data=df_cluster, x='depth_bgl_in_m', kde=True, ax=ax_depth, color='brown')
        ax_depth.set_title(f"Tiefe [m] (Durchschnitt: {df_cluster['depth_bgl_in_m'].mean():.1f})", fontsize=9)
        ax_depth.set_xlabel("")

    
    # ------------------------- Gesteinsarten (Reihe 2) -------------------------
    ax_bar = fig.add_subplot(gs[2])
    if 'rock_type' in df_cluster.columns:
        rock_counts = df_cluster['rock_type'].value_counts()
        new_labels = [str(l) for l in rock_counts.index]
        # ------------------------- Colorset 3 -------------------------
        unique_rocks_all = sorted(df_run['rock_type'].dropna().unique())
        pal = sns.color_palette('Set3', n_colors=12)
        rock_color_map = {rt: pal[i % 12] for i, rt in enumerate(unique_rocks_all)}
        
        # ------------------------- Labels in der Map -------------------------
        if len(new_labels) > 0:
            sns.barplot(x=new_labels, y=rock_counts.values, ax=ax_bar, hue=new_labels, palette=rock_color_map, legend=False)
            ax_bar.set_title("Gesteinsarten (Lokal)", fontsize=10)
            ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=45, ha='right')
            ax_bar.tick_params(axis='x', labelsize=8)

    # ------------------------- Chemische Verteilung (Reihe 3) -------------------------
    ax_chem_container = fig.add_subplot(gs[4])
    ax_chem_container.axis('off')
    ax_chem_container.set_title("Lokale Feature Verteilung (Originalwerte)", fontsize=12, pad=30)
    
    param_cols = train_cols[:12]
    n_plots = len(param_cols)
    
    if n_plots > 0:
        cols = 3
        rows = int(np.ceil(n_plots / cols))
        
        sub_gs = gs[4].subgridspec(rows, cols, hspace=0.5, wspace=0.3) 
        
        for i, col in enumerate(param_cols):
            row = i // cols
            col_idx = i % cols
            ax_sub = fig.add_subplot(sub_gs[row, col_idx])
            
            original_col_candidate = col.split('_log')[0].split('_gauss')[0].split('_boxcox')[0]
            
            plot_col = col 
            is_original = False
            if original_col_candidate in df_cluster.columns:
                plot_col = original_col_candidate
                is_original = True
            
            data_clean = df_cluster[plot_col].dropna()
            if len(data_clean) > 1:
                sns.histplot(data_clean, kde=True, ax=ax_sub, color='teal', element="step")
                
                title_text = plot_col.split('_in_')[0]
                if is_original:
                    if "_in_" in plot_col:
                        unit = plot_col.split('_in_')[1]
                        title_text += f" [{unit}]"
                else:
                    title_text += " (trans.)"
                
                # ------------------------- Clipping an Rändern -------------------------
                q995 = data_clean.quantile(0.995)
                right_limit = None
                if q995 > 0:
                     right_limit = q995 * 1.1
                
                ax_sub.set_xlim(left=0, right=right_limit)
                    
                ax_sub.set_title(title_text, fontsize=8)
                ax_sub.set_xlabel("")
                ax_sub.set_ylabel("")
            
            
    # ------------------------- Stratigraphic Periods (Reihe 4) -------------------------
    ax_footer = fig.add_subplot(gs[5])
    ax_footer.axis('off')
    strat_text = "Keine Stratigraphie Daten"
    if 'stratigraphic_period' in df_cluster.columns:
        uniques = df_cluster['stratigraphic_period'].dropna().unique()
        if len(uniques) > 0:
            strat_text = "Stratigraphic Periods: " + ", ".join([str(s) for s in uniques])
        
    ax_footer.text(0.5, 0.5, strat_text, ha='center', va='center', fontsize=9, wrap=True)

            
    plt.tight_layout(rect=[0, 0.02, 1, 0.98])
    pdf_obj.savefig(fig)
    plt.close(fig)


In [9]:
def create_component_planes_page(pdf_obj, df_run, train_cols, som_x, som_y):

    # ------------------------- Anzahl Plots -------------------------
    n_all = len(train_cols)
    
    # ------------------------- Layout Logik -------------------------
    # <= 6: 2x3 6 Diagramm pro Seite
    # 7-8: 2x4 8 Diagramme pro Seite
    # 9-12: 2x3 6 Diagramme je Seite, 2 Seiten
    # > 12: 2x4 8 Diagramme je Seite, 2 Seiten
    
    if n_all <= 6:
        cols = 2
        rows_grid = 3
        max_per_page = 6
    elif n_all <= 8:
        cols = 2
        rows_grid = 4
        max_per_page = 8
    elif n_all <= 12:
        cols = 2
        rows_grid = 3
        max_per_page = 6
    else:
        cols = 2
        rows_grid = 4
        max_per_page = 8

    # Anzahl Seiten berechnen
    n_pages = (n_all + max_per_page - 1) // max_per_page

    for p in range(n_pages):
        start_idx = p * max_per_page
        end_idx = min(start_idx + max_per_page, n_all)
        current_batch = train_cols[start_idx:end_idx]
        
        # ------------------------- A4 Layout -------------------------
        fig = plt.figure(figsize=(11.7, 16.5)) 

        # ------------------------- Layout mit Header -------------------------
        # rows_grid ist fix für Konsistenz über Seiten hinweg
        gs = fig.add_gridspec(rows_grid + 1, cols, height_ratios=[0.05] + [1.0/rows_grid]*rows_grid)
        

        # ------------------------- Titel -------------------------
        ax_title = fig.add_subplot(gs[0, :])
        ax_title.axis('off')
        
        page_suffix = ""
        if n_pages > 1:
            page_suffix = f" (Seite {p+1}/{n_pages})"
            
        ax_title.text(0.5, 0.5, f"Globale Komponentenebene{page_suffix}\n(Durchschnittswert - Originale Einheiten)", 
                      ha='center', va='center', fontsize=16, fontweight='bold')
                      
        for i, feature in enumerate(current_batch):
            r = (i // cols) + 1
            c = i % cols
            ax = fig.add_subplot(gs[r, c])
            
            # ------------------------- Finde Original-Spalte -------------------------
            # Versuche Boxcox / Log / Gauss Suffixe zu entfernen
            original_col_candidate = feature.split('_log')[0].split('_gauss')[0].split('_boxcox')[0]
            
            plot_col = feature
            if original_col_candidate in df_run.columns:
                plot_col = original_col_candidate
            
            # ------------------------- Grid berechnen -------------------------
            grid = np.full((som_x, som_y), np.nan)
            agg = df_run.groupby(['som_x', 'som_y'])[plot_col].mean()
            for (gx, gy), val in agg.items():
                if gx < som_x and gy < som_y:
                    grid[gx, gy] = val
            

            # ------------------------- Name bereinigen für Titel -------------------------
            if "_in_" in plot_col:
                base, unit = plot_col.split("_in_")
                clean_name = f"{base} [{unit}]"
            else:
                clean_name = plot_col
            
            # ------------------------- Plotten -------------------------
            collection = plot_hex_map_to_axis(ax, grid, clean_name, annotate=False, cmap='viridis')
            

            # ------------------------- [1,1] Top-Left Logik ----------------------------
            ax.invert_yaxis()
            
            # ---------------------------- Beschriftung: Colorbar -------------------------
            cbar = plt.colorbar(collection, ax=ax, orientation='vertical', fraction=0.046, pad=0.04)
            cbar.ax.tick_params(labelsize=6)
            
        plt.tight_layout()
        pdf_obj.savefig(fig)
        plt.close(fig)


In [10]:
def create_density_comparison_page(pdf_obj, df_run, train_cols, som_x, som_y):
    import matplotlib.ticker as ticker
    
    # ------------------------- Original Features ermitteln -------------------------
    original_features = []
    for col in train_cols:
        clean = col.split('_log')[0].split('_gauss')[0].split('_boxcox')[0]
        if clean in df_run.columns:
            if clean not in original_features:
                original_features.append(clean)
        else:
            original_features.append(col)
    
    # ------------------------- Zusatz Features (Temp & Tiefe) immer dazu -------------------------
    extras = ['temperature_in_c', 'depth_bgl_in_m']
    for e in extras:
        if e in df_run.columns:
            if e not in original_features:
                original_features.append(e)
            
    if not original_features: return
    
    # ------------------------- Layout Konfiguration & Seitenaufteilung -------------------------
    n_all = len(original_features)

    # Logik: 
    # <=6: 2x3 Diagramm-Anordnung
    # 7-8: 2x4 Diagramm-Anordnung
    # >8:  pro Seite Bestimmung je Fall, aufgeteilt auf zwei Seiten
    
    if n_all <= 6:
        cols = 2
        rows_grid = 3
        max_per_page = 6
    elif n_all <= 8:
        cols = 2
        rows_grid = 4
        max_per_page = 8
    elif n_all <= 12:
        cols = 2
        rows_grid = 3
        max_per_page = 6
    else:
        cols = 2
        rows_grid = 4
        max_per_page = 8

    n_pages = (n_all + max_per_page - 1) // max_per_page

    # ------------------------- Farbpalette für Cluster -------------------------
    cluster_labels = []
    for y in range(som_y):
        for x in range(som_x):
            cluster_labels.append(f"[{y+1},{x+1}]")
            
    n_clusters = len(cluster_labels)
    palette = sns.color_palette("tab20", n_colors=n_clusters)
    color_map = {lbl: palette[i] for i, lbl in enumerate(cluster_labels)}


    for p in range(n_pages):
        start_idx = p * max_per_page
        end_idx = min(start_idx + max_per_page, n_all)
        current_batch = original_features[start_idx:end_idx]
        
        fig = plt.figure(figsize=(11.7, 16.5)) 
        gs = fig.add_gridspec(rows_grid + 1, cols, height_ratios=[0.05] + [1.0/rows_grid]*rows_grid, hspace=0.5, wspace=0.3)
        
        # ------------------------- Header Bereich -------------------------
        ax_title = fig.add_subplot(gs[0, :])
        ax_title.axis('off')
        
        page_suffix = ""
        if n_pages > 1:
            page_suffix = f" (Seite {p+1}/{n_pages})"
            
        ax_title.text(0.5, 0.5, f"Globale Feature Verteilung (KDE Trendlinien){page_suffix}\n(Logarithmisch skaliert, Ausnahmen: pH und Temperatur)", 
                      ha='center', va='center', fontsize=16, fontweight='bold')
                      
        
        # ------------------------- Plot-Schleife für aktuelle Seite -------------------------
        for i, feature in enumerate(current_batch):
            r = (i // cols) + 1
            c = i % cols
            ax = fig.add_subplot(gs[r, c])
            
            # ------------------------- Titel und Einheit bestimmen -------------------------
            unit = ""
            if "_in_" in feature:
                base_name, unit = feature.split('_in_')
                title = f"{base_name} [{unit}]"
            else:
                title = feature
                
            has_data = False
            
            # ------------------------- Log-Skalierung Logik -------------------------
            use_log = True
            
            # ------------------------- Ausnahme: pH ist bereits log -> Linear -------------------------
            if "pH" in feature or feature == "pH":
                 use_log = False

            # ------------------------- Ausnahme: Temperatur soll linear sein -------------------------
            if "temperature" in feature:
                 use_log = False
            
            # ------------------------- Daten plotten pro Cluster -------------------------
            for y in range(som_y):
                for x in range(som_x):
                    lbl = f"[{y+1},{x+1}]"
                    subset = df_run[(df_run['som_x'] == x) & (df_run['som_y'] == y)]
                    vals = subset[feature].dropna()
                    
                    if len(vals) > 2:
                        try:
                            if use_log:
                                # ------------------------- Filtern <= 0 für Log-Skala -------------------------
                                vals_log = vals[vals > 0]
                                if len(vals_log) > 2:
                                    sns.kdeplot(vals_log, ax=ax, color=color_map[lbl], linewidth=2.0, 
                                                label=lbl, warn_singular=False, cut=0, common_norm=False, log_scale=True)
                                    has_data = True
                            else:
                                # ------------------------- Linearer Plot (pH, Temp) -------------------------
                                sns.kdeplot(vals, ax=ax, color=color_map[lbl], linewidth=2.0, 
                                            label=lbl, warn_singular=False, cut=0, common_norm=False)
                                has_data = True
                        except Exception as e:
                            pass
            
            # ------------------------- Achsen und Beschriftungen -------------------------
            if has_data:
                ax.set_title(title, fontsize=10, fontweight='bold')
                ax.set_ylabel("Dichte")
                
                if use_log:
                    ax.set_xscale('log')
                    # ------------------------- Spezifische Achsenbeschriftung für Tiefe -------------------------
                    if "depth" in feature:
                        ax.set_xlabel("m")
                        # ------------------------- Setze Skalar-Formatierung statt 10^x -------------------------
                        ax.xaxis.set_major_formatter(ticker.ScalarFormatter())
                        ax.ticklabel_format(style='plain', axis='x')
                else:
                    # ------------------------- Labels für nicht-logarithmische Plots -------------------------
                    if "pH" in feature:
                        ax.set_xlabel("pH")
                    elif "temperature" in feature:
                        ax.set_xlabel("Temperatur (°C)")
                    else:
                        ax.set_xlabel("Original Skala")
                
                ax.grid(True, which='both', linestyle='--', linewidth=0.5, alpha=0.7)
                
                # ------------------------- Legende anzeigen -------------------------
                ax.legend(fontsize=5, loc='upper left', ncol=2)
                
            else:
                ax.text(0.5,0.5, "Keine Daten (>0)", ha='center')
                
        plt.tight_layout()
        pdf_obj.savefig(fig)
        plt.close(fig)


In [11]:
def run_som_analysis(selection_names, run_id_suffix=""):

    # ----------------------- Mapping -----------------------
    train_cols, report = get_training_features(selection_names, df_full.columns)
    print(f"\n--- Start Run: {run_id_suffix} ---")
    print("Mapping:", report)
    if not train_cols:
        print("Keine validen Features. Skip.")
        return
        
    # ----------------------- Dateb filtern -----------------------
    analysis_cols = [c for c in selection_names if c in df_full.columns]
    mandatory = list(set(train_cols + analysis_cols + ['temperature_in_c', 'rock_type', 'pH', 'depth_bgl_in_m', 'stratigraphic_period']))
    mandatory = [c for c in mandatory if c in df_full.columns]
    
    df_run = df_full[mandatory].copy()
    
    # ----------------------- Temperaturfilter >= 10°C -----------------------
    if 'temperature_in_c' in df_run.columns:
         df_run['temperature_in_c'] = pd.to_numeric(df_run['temperature_in_c'], errors='coerce')
         cond = (df_run['temperature_in_c'] >= 10) | (df_run['temperature_in_c'].isna())
         df_run = df_run[cond]
    
    # ----------------------- Filter für komplette Daten -----------------------
    df_run.dropna(subset=train_cols, inplace=True)
    if len(df_run) < 50:
        print(f"Zu wenig Daten ({len(df_run)}). Skip.")
        return
        
    # ----------------------- Skalierung -----------------------
    scaler = MinMaxScaler(feature_range=(0, 1))
    data_scaled = scaler.fit_transform(df_run[train_cols].values)

    # ----------------------- SOM trainieren (Konfiguriert über Globale Parameter) -----------------------
    som_x = SOM_X
    som_y = SOM_Y
    
    som = MiniSom(x=som_x, y=som_y, input_len=len(train_cols), sigma=SOM_SIGMA, learning_rate=SOM_LEARNING_RATE, 
                  topology=SOM_TOPOLOGY, neighborhood_function=SOM_NEIGHBORHOOD, activation_distance=SOM_ACTIVATION, random_seed=SOM_RANDOM_SEED)
    som.random_weights_init(data_scaled)
    som.train_random(data_scaled, SOM_ITERATIONS)
    
    #  ---------------------- Analyse ----------------------
    winner_coords = [som.winner(x) for x in data_scaled]
    df_run['som_x'] = [c[0] for c in winner_coords]
    df_run['som_y'] = [c[1] for c in winner_coords]
    
    # ------------------------- Temperatur Karte (Seite 2) -------------------------
    temp_map = np.full((som_x, som_y), np.nan)
    if 'temperature_in_c' in df_run.columns:
         agg = df_run.groupby(['som_x', 'som_y'])['temperature_in_c'].mean()
         for (gx, gy), val in agg.items(): temp_map[gx, gy] = val
         

    # ------------------------- PDF exportieren -------------------------
    time_str = datetime.now().strftime("%H-%M-%S")
    clean_names = [n.split('_in_')[0] for n in selection_names]
    combo_str = "-".join(clean_names)[:50]
    
    # ------------------------- Output Pfad -------------------------
    out_dir = base_dir / "MiniSom_Results" / SESSION_TIMESTAMP
    out_dir.mkdir(parents=True, exist_ok=True)
    safe_combo = combo_str.replace("/", "_").replace("\\", "_")
    pdf_path = out_dir / f"Report_{run_id_suffix}_{safe_combo}.pdf"
    
    with PdfPages(pdf_path) as pdf:

        # ------------------------- Seite 1: Deckblatt -------------------------
        plt.figure(figsize=(10,6))
        plt.text(0.5, 0.5, f"Durchlauf: {run_id_suffix}\nFeatures: {', '.join(clean_names)}\nVollständige Zeilen: {len(df_run)}\nSOM Größe: {som_x}x{som_y}", ha='center', fontsize=12)
        plt.axis('off')
        pdf.savefig()
        plt.close()
        
        # ------------------------- Seite 2: Temperatur Karte mit Nummerierung ----------------------------
        f, ax = plt.subplots(figsize=(8,8))
        if 'temperature_in_c' in df_run.columns:

             # ------------------------- Manuelle Annotation, um Format ab [1,1] zu erreciehn -------------------------
             col_t = plot_hex_map_to_axis(ax, temp_map, "Durchschnittstemperatur Übersicht", annotate=False, cmap='coolwarm')
             
             # ------------------------- Annotation Loop für [x,y] Labels -------------------------
             for y in range(som_y):
                for x in range(som_x):
                     offset = 0.5 if y % 2 != 0 else 0.0
                     center_x = x + offset
                     center_y = y * (np.sqrt(3) / 2)
                     # ------------------------- Label [Zeile, Spalte] -> [y+1, x+1]
                     idx_label = f"[{y+1},{x+1}]"
                     # ------------------------- Temperatur Wert
                     val = temp_map[y, x]
                     val_str = f"{val:.1f}" if not pd.isna(val) else "-"
                     
                     ax.text(center_x, center_y+0.15, idx_label, ha='center', va='center', fontsize=6, fontweight='bold', color='black')
                     ax.text(center_x, center_y-0.15, val_str, ha='center', va='center', fontsize=6, color='blue')

             plt.colorbar(col_t, ax=ax, label="Durchschnittstemperatur [°C]")
             f.text(0.5, 0.05, "Lesehilfe: [Zeile, Spalte]. Blau = Temp.", ha='center')

             ax.invert_yaxis()
             
        pdf.savefig(f)
        plt.close(f)
        

        # ------------------------- Seite 3: Gesteinstyp Übersicht ----------------------------
        if 'rock_type' in df_run.columns:
            f_rock, ax_rock = plt.subplots(figsize=(8,8))
            

            # ------------------------- Dominierende Gesteinstypen herausfinden -------------------------
            dom_rocks = df_run.groupby(['som_x', 'som_y'])['rock_type'].apply(lambda x: x.mode().iloc[0] if not x.mode().empty else None)
            
            # ------------------------- Farbschema 3 verwenden -------------------------
            unique_rocks = df_run['rock_type'].dropna().unique()
            unique_rocks = sorted(unique_rocks)
            
            pal = sns.color_palette('Set3', n_colors=12)


            # ------------------------- Gesteinstypen zu Farbschmea -------------------------
            rock_color_map = {rt: pal[i % 12] for i, rt in enumerate(unique_rocks)}
            
            legend_handles = []
            seen_rocks = set()
            
            ax_rock.set_aspect('equal')

            for y in range(som_y):
                for x in range(som_x):
                    
                    rock = dom_rocks.get((x, y), None)
                    
                    offset = 0.5 if y % 2 != 0 else 0.0
                    center_x = x + offset
                    center_y = y * (np.sqrt(3) / 2)
                    radius = (1 / np.sqrt(3)) * 0.95
                    
                    fc = 'lightgrey'
                    if rock and rock in rock_color_map:
                        fc = rock_color_map[rock]
                        if rock not in seen_rocks:
                             legend_handles.append(mpatches.Patch(color=fc, label=str(rock)))
                             seen_rocks.add(rock)
                    
                    hex_poly = mpatches.RegularPolygon((center_x, center_y), numVertices=6, radius=radius,
                                                       orientation=np.radians(30), 
                                                       facecolor=fc, edgecolor='k', linewidth=0.5, alpha=0.9)
                    ax_rock.add_patch(hex_poly)
                    
                    idx_label = f"[{y+1},{x+1}]"
                    rock_label = str(rock) if rock else "-"
                    if len(rock_label) > 10: rock_label = rock_label[:9] + "."
                    
                    ax_rock.text(center_x, center_y+0.15, idx_label, ha='center', va='center', fontsize=7, fontweight='bold', color='black')
                    ax_rock.text(center_x, center_y-0.15, rock_label, ha='center', va='center', fontsize=6, color='black')

            ax_rock.set_xlim(-0.5, som_x + 0.5)
            ax_rock.set_ylim(-0.5, som_y * (np.sqrt(3)/2) + 0.5)
            ax_rock.invert_yaxis()
            ax_rock.axis('off')
            ax_rock.set_title("Dominante Gesteinsarten Übersicht", fontsize=14)
            
            ax_rock.legend(handles=legend_handles, loc='upper center', bbox_to_anchor=(0.5, -0.05), ncol=3, fontsize=8)
            
            pdf.savefig(f_rock)
            plt.close(f_rock)




        create_component_planes_page(pdf, df_run, train_cols, som_x, som_y)

        
        # ------------------------- Seite 3 bis x-1: Cluster Details -------------------------
        # Loop über alle Cluster [1,1] bis [x,y]
        for iy in range(som_y):
            for ix in range(som_x):
                # -------------------------- Daten filtern -------------------------
                mask = (df_run['som_x'] == ix) & (df_run['som_y'] == iy)
                df_cluster = df_run[mask]
                
                
                #  ------------------------- Seite erzeugen -------------------------
                create_cluster_page(pdf, df_cluster, (iy+1, ix+1), train_cols, som_x, som_y, df_run)
             
        # ------------------------- Seite X: Globale Dichteverteilung -------------------------
        create_density_comparison_page(pdf, df_run, train_cols, som_x, som_y)
    print(f"Report saved: {pdf_path}")


In [ ]:
# ------------------------------- MANUELLE AUSFÜHRUNG -------------------------------

if EXECUTION_MODE == 'MANUAL':

    print("\n=== Modus: MANUAL ===")

    # ---------------------- Kombiniere Basis + Alle Optionalen ----------------------
    full_selection = FIXED_BASE_FEATURES + OPTIONAL_FEATURES_POOL
    # ---------------------- Duplikate entfernen ----------------------
    full_selection = list(set(full_selection))
    
    print(f"Starte Analyse für {len(full_selection)} Features...")
    run_som_analysis(full_selection, "Manual_Run")
    

# ------------------------------- LOOP AUSFÜHRUNG -------------------------------

elif EXECUTION_MODE == 'LOOP':

    print("\n=== Modus: LOOP ===")
    
    base = FIXED_BASE_FEATURES
    pool = OPTIONAL_FEATURES_POOL
    
    # ------------------------- nur mit Basis als Referenz -------------------------
    combos_to_run = [base]
    labels = ["Base_without_pH"]
    
    # ------------------------- alle Kombinationen -------------------------
    max_r = len(pool)
    if max_r > 8:
        print("Zu viele optionale Features für vollen Loop. Beschränke auf max 3 Zusatz-Attribute.")
        max_r = 3
        
    for r in range(1, max_r + 1):
        for subset in itertools.combinations(pool, r):
            # ------------------------- Neue Features = Basis + Subset -------------------------
            combined = list(base) + list(subset)
            combos_to_run.append(combined)
            
            # ------------------------- Label für Ordnernamen -------------------------
            addon_names = [n.split('_in_')[0] for n in subset]
            lbl_temp = "Plus_" + "-".join(addon_names)
            if lbl_temp == "Plus_pH":
                lbl_temp = "Base_with_pH"
            labels.append(lbl_temp)


    # ------------------------- MANUELLE ZUSATZ-KOMBINATION -------------------------
    combos_to_run.append(["pH", "K_in_mmol/L", "Fe_in_mmol/L", "Mn_in_mmol/L"])
    labels.append("Plus_pH-K-Fe-Mn")

    print(f"{len(combos_to_run)} Runs geplant.")

    # ------------------------- Sortieruung nach kompletten Zeilen (automatische mehr Features desto später) -------------------------
    print("Berechne Datenverfügbarkeit für Sortierung...")
    
    combo_stats = []
    

    for idx, combo in enumerate(combos_to_run):
        lbl = labels[idx]
        
        # ------------------------- Farbe feststellen -------------------------
        c_train, _ = get_training_features(combo, df_full.columns)
        
        # ------------------------- Base only immer am Anfang -------------------------
        if lbl == "Base_without_pH":
             count = 999999999
        elif lbl == "Base_with_pH":
             count = 999999998
        elif lbl == "Plus_pH-Fe":
             count = 999999997
        elif lbl == "Plus_pH-K-Fe":
             count = 999999996
        elif lbl == "Plus_pH-K-Fe-Mn":
             count = 999999995
        elif lbl == "Plus_temperature":
             count = 999999994
        else:

            #  ------------------------ Filterlogik -------------------------
            c_analysis = [c for c in combo if c in df_full.columns]
            mandatory = list(set(c_train + c_analysis + ['temperature_in_c']))
            mandatory = [c for c in mandatory if c in df_full.columns]
            
            # ------------------------- Schnellcheck -------------------------
            temp_df = df_full[mandatory].copy()
            if 'temperature_in_c' in temp_df.columns:
                 temp_df['temperature_in_c'] = pd.to_numeric(temp_df['temperature_in_c'], errors='coerce')
                 temp_df = temp_df[(temp_df['temperature_in_c'] >= 10) | (temp_df['temperature_in_c'].isna())]
            
            temp_df.dropna(subset=c_train, inplace=True)
            count = len(temp_df)
            
        combo_stats.append({
            'combo': combo,
            'label': lbl,
            'Anzahl': count
        })

    combo_stats.sort(key=lambda x: x['Anzahl'], reverse=True)
    
    combos_to_run = [item['combo'] for item in combo_stats]
    labels = [item['label'] for item in combo_stats]
    
    print("Sortierung abgeschlossen.")

    # ------------------------- LIMITIERUNG DURCH PIPELINE -------------------------
    if 'SOM_LOOP_LIMIT' in os.environ:
        try:
            loop_limit = int(os.environ['SOM_LOOP_LIMIT'])
            print(f"Limitiere Loop auf {loop_limit} Durchlaeufe (durch SOM_LOOP_LIMIT).")
            combos_to_run = combos_to_run[:loop_limit]
            labels = labels[:loop_limit]
        except ValueError:
            print(f"Warnung: Ungueltiges SOM_LOOP_LIMIT: {os.environ['SOM_LOOP_LIMIT']}")

    # ------------------------- Umbenennung des ersten Runs (Base_without_pH -> Test-Run) -------------------------
    if len(labels) > 0 and labels[0] == "Base_without_pH":
        labels[0] = "Test-Run"

    # ------------------------- Spacer einfuegen: 001_Basis_Durchlaeufe -------------------------
    if len(combos_to_run) >= 1:
        combos_to_run.insert(1, ["SEPARATOR"])
        labels.insert(1, "Basis_Durchlaeufe")

    # ------------------------- Separator einfuegen (Zusatzinfos) -------------------------
    if len(combos_to_run) >= 6:
        print("Fuege Separator an Position 5 ein")
        combos_to_run.insert(6, ["SEPARATOR"])
        labels.insert(6, "Zusatzinformationen")
    
    for i, combination in enumerate(combos_to_run):
        run_label = labels[i]
        # ------------------------- ID generieren (0-based -> 000, 001, ...) -------------------------
        run_id = f"{i:03d}_{run_label}"
        
        print(f"\n--- Loop Schritt {i}/{len(combos_to_run)}: {run_label} ---")
        
        # ------------------------- Separator Check -------------------------
        if combination == ["SEPARATOR"]:
             print(f"Erstelle Trennlinie: {run_id}")
             # ------------------------- Zieldatei erstellen
             out_dir_sep = base_dir / "MiniSom_Results" / SESSION_TIMESTAMP
             out_dir_sep.mkdir(parents=True, exist_ok=True)
             sep_path = out_dir_sep / f"Report_{run_id}_Trennlinie.txt"
             with open(sep_path, "w") as f:
                 f.write("Dies ist eine Trennlinie zwischen den priorisierten Runs und dem Rest.")
             continue
        run_som_analysis(combination, run_id)

else:
    print(f"Unbekannter Modus: {EXECUTION_MODE}")



=== Modus: LOOP ===
Zu viele optionale Features für vollen Loop. Beschränke auf max 3 Zusatz-Attribute.
697 Runs geplant.
Berechne Datenverfügbarkeit für Sortierung...
Sortierung abgeschlossen.
Limitiere Loop auf 50 Durchlaeufe (durch SOM_LOOP_LIMIT).
Fuege Separator an Position 5 ein

--- Loop Schritt 0/52: Test-Run ---

--- Start Run: 000_Test-Run ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'total_dissolved_solids_in_mmol/L -> total_dissolved_solids_in_mmol/L_log_gauss']


C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:168: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=45, ha='right')
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.02, 1, 0.98])
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:168: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=45, ha='right')
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.02, 1, 0.98])
C:\Use

Report saved: C:\Users\lucca\Desktop\Abschlussarbeit HTWK\abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-17_15-05-11\Report_000_Test-Run_Na-Mg-Ca-Cl-SO4-HCO3-total_dissolved_solids.pdf

--- Loop Schritt 1/52: Basis_Durchlaeufe ---
Erstelle Trennlinie: 001_Basis_Durchlaeufe

--- Loop Schritt 2/52: Base_with_pH ---

--- Start Run: 002_Base_with_pH ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'total_dissolved_solids_in_mmol/L -> total_dissolved_solids_in_mmol/L_log_gauss', 'pH -> pH_gauss']


C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:168: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=45, ha='right')
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.02, 1, 0.98])
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:168: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=45, ha='right')
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.02, 1, 0.98])
C:\Use

Report saved: C:\Users\lucca\Desktop\Abschlussarbeit HTWK\abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-17_15-05-11\Report_002_Base_with_pH_Na-Mg-Ca-Cl-SO4-HCO3-total_dissolved_solids-pH.pdf

--- Loop Schritt 3/52: Plus_pH-Fe ---

--- Start Run: 003_Plus_pH-Fe ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'total_dissolved_solids_in_mmol/L -> total_dissolved_solids_in_mmol/L_log_gauss', 'pH -> pH_gauss', 'Fe_in_mmol/L -> Fe_in_mmol/L_log_gauss']


...

Report saved: C:\Users\lucca\Desktop\Abschlussarbeit HTWK\abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-17_15-05-11\Report_049_Plus_temperature-F_Na-Mg-Ca-Cl-SO4-HCO3-total_dissolved_solids-temper.pdf

--- Loop Schritt 50/52: Plus_temperature-K-F ---

--- Start Run: 050_Plus_temperature-K-F ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'total_dissolved_solids_in_mmol/L -> total_dissolved_solids_in_mmol/L_log_gauss', 'temperature_in_c -> temperature_in_c_log_gauss', 'K_in_mmol/L -> K_in_mmol/L_log_gauss', 'F_in_umol/L -> F_in_umol/L_log_gauss']


C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.02, 1, 0.98])
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:168: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=45, ha='right')
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.02, 1, 0.98])
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:168: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=45, ha='right')
C:\Use

Report saved: C:\Users\lucca\Desktop\Abschlussarbeit HTWK\abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-17_15-05-11\Report_050_Plus_temperature-K-F_Na-Mg-Ca-Cl-SO4-HCO3-total_dissolved_solids-temper.pdf

--- Loop Schritt 51/52: Plus_temperature-K-depth_bgl ---

--- Start Run: 051_Plus_temperature-K-depth_bgl ---
Mapping: ['Na_in_mmol/L -> Na_in_mmol/L_log_gauss', 'Mg_in_mmol/L -> Mg_in_mmol/L_log_gauss', 'Ca_in_mmol/L -> Ca_in_mmol/L_log_gauss', 'Cl_in_mmol/L -> Cl_in_mmol/L_log_gauss', 'SO4_in_mmol/L -> SO4_in_mmol/L_log_gauss', 'HCO3_in_mmol/L -> HCO3_in_mmol/L_log_gauss', 'total_dissolved_solids_in_mmol/L -> total_dissolved_solids_in_mmol/L_log_gauss', 'temperature_in_c -> temperature_in_c_log_gauss', 'K_in_mmol/L -> K_in_mmol/L_log_gauss', 'depth_bgl_in_m -> depth_bgl_in_m_gauss']


C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:168: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=45, ha='right')
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.02, 1, 0.98])
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:168: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax_bar.set_xticklabels(ax_bar.get_xticklabels(), rotation=45, ha='right')
C:\Users\lucca\AppData\Local\Temp\ipykernel_2280\2394761395.py:235: UserWarning: This figure includes Axes that are not compatible with tight_layout, so results might be incorrect.
  plt.tight_layout(rect=[0, 0.02, 1, 0.98])
C:\Use

Report saved: C:\Users\lucca\Desktop\Abschlussarbeit HTWK\abschlussarbeit\Abschlussarbeit Bearbeitung\Jupyter Notebooks\3_Machine-Learning\3.2_Machine-Learning\MiniSom\MiniSom_Results\2026-01-17_15-05-11\Report_051_Plus_temperature-K-depth_bgl_Na-Mg-Ca-Cl-SO4-HCO3-total_dissolved_solids-temper.pdf
